# NimbusPay ticket triage: LoRA fine-tune of `Qwen/Qwen2.5-3B-Instruct`

**Final model:** Qwen2.5-3B-Instruct loaded in **fp16 (not 4-bit)** + LoRA r=16 (29.9M trainable), trained on the
cleaned train set plus 370 rule-boundary augmentations. Dev (200): ~93–95% exact match, ~99% mean field accuracy.

**This notebook runs top to bottom on a fresh Colab T4 (~1 h 45 min)** and reproduces the final model and
`predictions_test.jsonl` from `train.jsonl`:

| § | What | Part | Runs here? |
|---|---|---|---|
| 0 | Setup | – | yes |
| 1 | Rebuild the cleaned train set from `train.jsonl` + `cleaning_log.csv`, then the augmentation | A, D | yes (CPU, ~2 min) |
| 2 | Chat template + token lengths | C | yes (CPU, ~1 min) |
| 3 | Baselines | B | shown from saved logs (code included) |
| 4 | Pipeline checks: loss mask, padding, worst-case memory | C | yes (~3 min) |
| 5 | Controlled experiments R1, D1–D4 + results table + paired tests | D | shown from saved logs (code included) |
| 6 | **Train the final model** (D4 configuration, from scratch) | final | yes (~45 min) |
| 7 | Error analysis | E | yes |
| 8 | Test predictions one row at a time (the graders' call) + reproducibility check | final | yes (~50 min) |
| 9 | Collect the files for the repository | – | yes |

The data audit itself (how each problem was found) is in `data_prep.ipynb`; it produced `cleaning_log.csv`.

**Interrupted?** Re-run all cells: training resumes from its last checkpoint and the test predictions skip rows
already written.

## 0. Setup

In [1]:
# 0.1 Pinned versions (all results were produced with these).
# Colab ships torchao 0.10, which PEFT 0.21 rejects when it creates LoRA layers; torchao is unused here.
!pip install -q transformers==5.22.2 peft==0.21.2 accelerate==1.15.0 bitsandbytes==0.50.2
!pip uninstall -y -q torchao

In [2]:
# 0.2 Environment
import importlib, sys, torch
for p in ['torch', 'transformers', 'peft', 'accelerate', 'bitsandbytes']:
    print(f'{p:14}', importlib.import_module(p).__version__)
print('python        ', sys.version.split()[0])
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
print('GPU           ', torch.cuda.get_device_name(0), '| capability', torch.cuda.get_device_capability(),
      '| native bf16:', torch.cuda.is_bf16_supported(including_emulation=False), '-> fp16 everywhere')

torch          2.14.1+cu130
transformers   5.22.2
peft           0.21.2
accelerate     1.15.0
bitsandbytes   0.50.2
python         3.13.16
GPU            Tesla T4 | capability (7, 5) | native bf16: False -> fp16 everywhere


In [3]:
# 0.3 Paths and data
import json, os, re, sys, gc, math, time, random, shutil, collections, glob
import numpy as np, pandas as pd
T_START = time.time()

USE_DRIVE = True        # True: work in Google Drive (survives disconnects). False: everything under /content/work
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/vima3ya'
else:
    ROOT = '/content/work'
# Expected under ROOT:
#   candidate_pack/      the assignment pack: data/{train,dev,test_inputs}.jsonl, SCHEMA.md, score.py
#   cleaning_log.csv     from the repository  (or prep_outputs/cleaning_log.csv written by data_prep.ipynb)
#   logs/                saved baseline/experiment results from the repository  (or runs/ from the original runs)
DATA = f'{ROOT}/candidate_pack/data'
CLEAN_LOG = next((p for p in [f'{ROOT}/cleaning_log.csv', f'{ROOT}/prep_outputs/cleaning_log.csv'] if os.path.exists(p)), None)
LOGS = next((p for p in [f'{ROOT}/logs', f'{ROOT}/runs'] if os.path.isdir(p)), None)
assert CLEAN_LOG, 'cleaning_log.csv not found under ROOT'
OUT  = f'{ROOT}/official'                      # everything this run produces
RUNS = f'{OUT}/runs'                           # this run's training folders (checkpoints -> resumable)
os.makedirs(RUNS, exist_ok=True)
REF_CLEAN = f'{ROOT}/prep_outputs/train_clean.jsonl'   # optional: data_prep.ipynb's output, to prove the rebuild matches
REF_AUG   = f'{ROOT}/prep_outputs/train_aug.jsonl'     # optional: exp4_augmentation.ipynb's output
print('cleaning log:', CLEAN_LOG, '| saved logs:', LOGS, '| outputs:', OUT)

def load_jsonl(p):
    out = []
    with open(p, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:           # a half-written last line after a disconnect
                    pass
    return out

train_raw = load_jsonl(f'{DATA}/train.jsonl')
dev       = load_jsonl(f'{DATA}/dev.jsonl')
test      = load_jsonl(f'{DATA}/test_inputs.jsonl')
gold      = {r['id']: json.loads(r['messages'][2]['content']) for r in dev}
dev_text  = {r['id']: r['messages'][1]['content'] for r in dev}
KEYS = ['category', 'priority', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human']
R1, D4 = 'R1_clean_lora16_fp16', 'D4_aug_lora16_fp16'
print(len(train_raw), len(dev), len(test))      # 1445 200 400

Mounted at /content/drive
cleaning log: /content/drive/MyDrive/vima3ya/prep_outputs/cleaning_log.csv | saved logs: /content/drive/MyDrive/vima3ya/runs | outputs: /content/drive/MyDrive/vima3ya/official
1445 200 400


In [4]:
# 0.4 Tokenizer, prompt format, encoding with the loss mask
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tok      = AutoTokenizer.from_pretrained(MODEL_ID)
SYSTEM   = dev[0]['messages'][0]['content']                   # the fixed system prompt (same in every split)
MAX_LEN  = 2304                                               # longest train row = 2,215 tokens -> nothing truncated
END_ID   = tok.convert_tokens_to_ids('<|im_end|>')            # end-of-turn token = eos: the model must learn it

def ids(text):
    # transformers v5: apply_chat_template(tokenize=True) returns a dict, so tokenize the rendered text instead
    return tok(text, add_special_tokens=False)['input_ids']

def build_prompt(msgs):
    """system + user with Qwen's own chat template, ending in '<|im_start|>assistant\n'."""
    return tok.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)

def encode(row):
    m = row['messages']
    p = ids(build_prompt(m))
    a = ids(m[2]['content'] + '<|im_end|>')       # answer + stop token; the trailing '\n' is never generated
    x = p + a
    assert len(x) <= MAX_LEN, f"{row['id']} would be truncated"
    return {'id': row['id'], 'input_ids': x, 'labels': [-100] * len(p) + a}   # loss only on answer + <|im_end|>

def encode_all(rows):
    enc = [encode(r) for r in rows]
    for r, e in zip(rows, enc):                    # our tokens must equal the chat template's tokens exactly
        full = ids(tok.apply_chat_template(r['messages'], tokenize=False))
        assert full[:len(e['input_ids'])] == e['input_ids'], r['id']
        assert e['labels'][-1] == END_ID
    return enc

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [5]:
# 0.5 GPU memory + checkpoint helpers
from peft import set_peft_model_state_dict, load_peft_weights

def free_gpu(*names):
    for n in (names or ('model', 'base', 'pm', 'opt', 'scaler', 'sched')):
        if n in globals():
            del globals()[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'allocated {torch.cuda.memory_allocated()/2**30:.2f} GiB | reserved {torch.cuda.memory_reserved()/2**30:.2f} GiB')

def save_ckpt(run_dir, pm, opt, scaler, sched, step, epoch):
    """Atomically write the latest checkpoint: adapter (PEFT format) + optimizer, scaler, scheduler, RNG states."""
    ck, tmp = f'{run_dir}/ckpt', f'{run_dir}/ckpt_tmp'
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    pm.save_pretrained(tmp)
    torch.save({'opt': opt.state_dict(), 'scaler': scaler.state_dict(), 'sched': sched.state_dict(),
                'step': step, 'epoch': epoch, 'torch_rng': torch.get_rng_state(),
                'cuda_rng': torch.cuda.get_rng_state(), 'py_rng': random.getstate()}, f'{tmp}/trainer_state.pt')
    if os.path.exists(ck):
        shutil.rmtree(ck)
    os.rename(tmp, ck)
    print(f'  [ckpt] saved step {step} (epoch {epoch})')

def load_ckpt(run_dir, pm, opt, scaler, sched):
    ck = f'{run_dir}/ckpt'
    if not os.path.exists(f'{ck}/trainer_state.pt'):
        return 0, 0
    set_peft_model_state_dict(pm, load_peft_weights(ck))
    st = torch.load(f'{ck}/trainer_state.pt', weights_only=False)
    opt.load_state_dict(st['opt']); scaler.load_state_dict(st['scaler']); sched.load_state_dict(st['sched'])
    torch.set_rng_state(st['torch_rng']); torch.cuda.set_rng_state(st['cuda_rng']); random.setstate(st['py_rng'])
    print(f'  [ckpt] resumed from step {st["step"]} (epoch {st["epoch"]})')
    return st['step'], st['epoch']

In [6]:
# 0.6 Length-grouped batching (less padding) + collate (right padding for training)
def length_grouped_batches(enc, bs, seed, mega=64):
    rng = random.Random(seed)
    idx = list(range(len(enc))); rng.shuffle(idx)
    batches = []
    for s in range(0, len(idx), mega):                       # shuffle, then sort inside chunks of 64
        chunk = sorted(idx[s:s + mega], key=lambda k: len(enc[k]['input_ids']), reverse=True)
        batches += [chunk[i:i + bs] for i in range(0, len(chunk), bs)]
    rng.shuffle(batches)
    return batches

def collate(enc, batch_idx):
    rows = [enc[k] for k in batch_idx]
    T = max(len(r['input_ids']) for r in rows)
    inp = torch.full((len(rows), T), tok.pad_token_id)       # pad (151643) != eos (151645)
    lab = torch.full((len(rows), T), -100)
    att = torch.zeros((len(rows), T), dtype=torch.long)
    for i, r in enumerate(rows):
        n = len(r['input_ids'])
        inp[i, :n], lab[i, :n], att[i, :n] = torch.tensor(r['input_ids']), torch.tensor(r['labels']), 1
    return {k: v.cuda() for k, v in {'input_ids': inp, 'labels': lab, 'attention_mask': att}.items()}

def padding_efficiency(enc, batches):
    real = sum(len(enc[k]['input_ids']) for b in batches for k in b)
    padded = sum(max(len(enc[k]['input_ids']) for k in b) * len(b) for b in batches)
    return real / padded

In [7]:
# 0.7 Model: base (fp16 or 4-bit NF4), LoRA, answer-only loss, inference loading, fp16 adapter export
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

BNB_4BIT = dict(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

def load_base(quant4=False):
    free_gpu()
    kw = dict(dtype=torch.float16, device_map='cuda', attn_implementation='sdpa')
    if quant4:
        kw['quantization_config'] = BitsAndBytesConfig(**BNB_4BIT)
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
    m.config.use_cache = False
    if quant4:
        m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={'use_reentrant': False})
    else:
        m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        m.enable_input_require_grads()
    return m

def add_lora(m, r=16, alpha=32, dropout=0.05, seed=0):
    torch.manual_seed(seed)
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, bias='none', task_type='CAUSAL_LM',
                     target_modules=LORA_TARGETS)
    pm = get_peft_model(m, cfg)
    n = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    print(f'trainable params: {n/1e6:.2f}M (cap 50M) | dtypes: {({p.dtype for p in pm.parameters() if p.requires_grad})}')
    assert n <= 50_000_000
    return pm

def answer_loss(pm, batch):
    """Cross-entropy on answer tokens only; logits computed only at those positions."""
    base = pm.get_base_model()
    h = base.model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'],
                   use_cache=False).last_hidden_state
    tgt  = batch['labels'][:, 1:]
    keep = tgt != -100
    logits = base.lm_head(h[:, :-1][keep]).float()
    return F.cross_entropy(logits, tgt[keep])

def load_for_inference(adapter_dir=None, quant4=False):
    """How the graders will load it: fp16 base model + our PEFT adapter."""
    free_gpu()
    kw = dict(dtype=torch.float16, device_map='cuda')
    if quant4:
        kw['quantization_config'] = BitsAndBytesConfig(**BNB_4BIT)
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
    if adapter_dir:
        m = PeftModel.from_pretrained(m, adapter_dir)
    return m.eval()

def export_fp16_adapter(src, dst):
    """PEFT adapter with weights cast to fp16: ~57 MB instead of ~115 MB (fits GitHub's 100 MB limit)."""
    from safetensors.torch import load_file, save_file
    os.makedirs(dst, exist_ok=True)
    sd = load_file(f'{src}/adapter_model.safetensors')
    save_file({k: v.to(torch.float16).contiguous() for k, v in sd.items()},
              f'{dst}/adapter_model.safetensors', metadata={'format': 'pt'})
    shutil.copy(f'{src}/adapter_config.json', f'{dst}/adapter_config.json')
    print(f'adapter exported in fp16: {os.path.getsize(f"{dst}/adapter_model.safetensors")/2**20:.0f} MB -> {dst}')

In [8]:
# 0.8 Generation + scoring (the graders' score.py) + paired comparisons
sys.path.insert(0, f'{ROOT}/candidate_pack')
import score as scorer
from scipy.stats import binomtest

def run_file(run, fname):
    """A saved file of `run`: this notebook's runs first, then the saved experiment logs."""
    for b in [RUNS, LOGS]:
        if b and os.path.exists(f'{b}/{run}/{fname}'):
            return f'{b}/{run}/{fname}'
    raise FileNotFoundError(f'{run}/{fname}')

@torch.no_grad()
def generate_outputs(rows, mdl, system=None, batch_size=16, **gen_kw):
    """Batched greedy generation (used for dev evaluation during training). Left padding."""
    mdl.eval()
    tok.padding_side = 'left'
    items = []
    for r in rows:
        m = [dict(r['messages'][0]), dict(r['messages'][1])]
        if system is not None:
            m[0]['content'] = system
        items.append((r['id'], build_prompt(m)))
    order = sorted(range(len(items)), key=lambda k: len(items[k][1]))
    outs, t0 = {}, time.time()
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tok([items[k][1] for k in idx], return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
        gen = mdl.generate(**enc, max_new_tokens=200, do_sample=False, use_cache=True, **gen_kw)
        for k, t in zip(idx, tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)):
            outs[items[k][0]] = t.strip()
    print(f'  generated {len(rows)} outputs in {time.time() - t0:.0f}s')
    return [{'id': r['id'], 'output': outs[r['id']]} for r in rows]

@torch.no_grad()
def predict_one(mdl, row, **gen_kw):
    """The graders' call: one row, the model's own chat template, greedy, max_new_tokens=200, strip only."""
    enc = tok(build_prompt(row['messages']), return_tensors='pt', add_special_tokens=False).to('cuda')
    out = mdl.generate(**enc, max_new_tokens=200, do_sample=False, **gen_kw)
    return tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def print_score(name, res):
    a = res['all']
    print(f"  [{name}] valid {a['json_valid']:.1%} | exact {a['exact_match']:.1%} | field {a['mean_field_acc']:.1%} | "
          + ' '.join(f"{k[:6]}={v:.0%}" for k, v in a['fields'].items()))

def evaluate(preds, name, gold_rows=None):
    gold_rows = gold_rows or dev
    d = f'{RUNS}/{name}'
    os.makedirs(d, exist_ok=True)
    with open(f'{d}/dev_pred.jsonl', 'w', encoding='utf-8') as f:
        for p in preds:
            f.write(json.dumps(p, ensure_ascii=False) + '\n')
    res = scorer.score(gold_rows, preds)
    json.dump(res, open(f'{d}/dev_score.json', 'w'), indent=1)
    print_score(name, res)
    return res

def append_result(row):
    p = f'{RUNS}/results.csv'
    df = pd.read_csv(p) if os.path.exists(p) else pd.DataFrame()
    if len(df):
        df = df[df['run'] != row['run']]
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(p, index=False)

def row_exact(run, ids=None):
    P = {p['id']: p['output'] for p in load_jsonl(run_file(run, 'dev_pred.jsonl'))}
    out = {}
    for i, g in gold.items():
        if ids is not None and i not in ids:
            continue
        p = scorer.parse(P.get(i))
        out[i] = p is not None and set(p) == set(scorer.KEYS) and all(scorer.same(p[k], g[k]) for k in scorer.KEYS)
    return out

def paired(a, b, ids=None):
    """Same dev rows: how many rows b fixed / broke relative to a, exact McNemar (binomial) test."""
    A, B = row_exact(a, ids), row_exact(b, ids)
    fixed = sum((not A[i]) and B[i] for i in A)
    broke = sum(A[i] and (not B[i]) for i in A)
    p = binomtest(min(fixed, broke), fixed + broke, 0.5).pvalue if fixed + broke else 1.0
    print(f'{a}  ->  {b}: exact {sum(A.values())}/{len(A)} -> {sum(B.values())}/{len(B)} | '
          f'fixed {fixed}, broke {broke} | McNemar p = {p:.3f}')

def score_subset(run, ids):
    return scorer.score([r for r in dev if r['id'] in ids], load_jsonl(run_file(run, 'dev_pred.jsonl')))['all']

def saved_score(run):
    return json.load(open(run_file(run, 'dev_score.json')))

In [9]:
# 0.9 Training loop: AdamW on the LoRA parameters, fp16 autocast + GradScaler, gradient accumulation, warmup + cosine,
#     checkpoint every 25 steps and every epoch (exact resume), dev evaluation after every epoch.
def cosine_with_warmup(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warmup) / max(1, total - warmup))))
    return f

def run_experiment(name, enc, *, quant4=False, r=16, alpha=None, lr=2e-4, epochs=3, bs=4, accum=4,
                   seed=0, warmup_frac=0.05, ckpt_every=25, eval_each_epoch=True, note=''):
    global base, pm, opt, scaler, sched
    run_dir = f'{RUNS}/{name}'
    os.makedirs(run_dir, exist_ok=True)
    json.dump(dict(name=name, quant4=quant4, r=r, alpha=alpha or 2 * r, lr=lr, epochs=epochs, bs=bs, accum=accum,
                   seed=seed, warmup_frac=warmup_frac, n_train=len(enc), note=note),
              open(f'{run_dir}/config.json', 'w'), indent=1)
    random.seed(seed); torch.manual_seed(seed)
    base = load_base(quant4)
    pm = add_lora(base, r=r, alpha=alpha or 2 * r, seed=seed)
    params = [p for p in pm.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    scaler = torch.amp.GradScaler('cuda')
    steps_per_epoch = math.ceil(math.ceil(len(enc) / bs) / accum)
    total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, cosine_with_warmup(total, max(1, int(warmup_frac * total))))

    start, _ = load_ckpt(run_dir, pm, opt, scaler, sched)
    log_path = f'{run_dir}/train_log.jsonl'
    t_train = 0.0
    if start and os.path.exists(log_path):
        t_train = load_jsonl(log_path)[-1]['train_s']
    torch.cuda.reset_peak_memory_stats()
    peak_train, step = 0.0, 0
    for ep in range(epochs):
        batches = length_grouped_batches(enc, bs, seed + ep)                 # same order on every resume
        groups = [batches[i:i + accum] for i in range(0, len(batches), accum)]
        for g in groups:
            if step < start:
                step += 1
                continue
            pm.train(); t0 = time.time(); tot = 0.0
            for b in g:
                with torch.autocast('cuda', dtype=torch.float16):
                    loss = answer_loss(pm, collate(enc, b)) / len(g)
                scaler.scale(loss).backward()
                tot += loss.item()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(params, 1.0).item()
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            step += 1
            t_train += time.time() - t0
            with open(log_path, 'a') as f:
                f.write(json.dumps({'step': step, 'epoch': ep, 'loss': tot, 'lr': sched.get_last_lr()[0],
                                    'grad_norm': gn, 'scale': scaler.get_scale(), 'train_s': t_train}) + '\n')
            if step % 10 == 0 or step == 1:
                print(f'  step {step:4d}/{total} | ep {ep+1} | loss {tot:.4f} | lr {sched.get_last_lr()[0]:.2e} '
                      f'| gnorm {gn:.2f} | scale {scaler.get_scale():.0f} | {t_train/60:.1f} min')
            if step % ckpt_every == 0 and step % steps_per_epoch != 0:
                save_ckpt(run_dir, pm, opt, scaler, sched, step, ep)
        peak_train = max(peak_train, torch.cuda.max_memory_allocated() / 2**30)
        if step > start or not os.path.exists(f'{run_dir}/ckpt'):
            save_ckpt(run_dir, pm, opt, scaler, sched, step, ep)
        if (eval_each_epoch or ep == epochs - 1) and not os.path.exists(f'{run_dir}/ep{ep+1}/dev_score.json'):
            evaluate(generate_outputs(dev, pm), f'{name}/ep{ep+1}')
            torch.cuda.reset_peak_memory_stats()
    pm.save_pretrained(f'{run_dir}/adapter')
    a = json.load(open(f'{run_dir}/ep{epochs}/dev_score.json'))['all']
    row = dict(run=name, note=note, quant='4-bit' if quant4 else '16-bit', r=r, lr=lr, epochs=epochs, n_train=len(enc),
               json_valid=round(a['json_valid'], 4), exact=round(a['exact_match'], 4),
               mean_field=round(a['mean_field_acc'], 4), peak_train_gib=round(peak_train, 2),
               train_min=round(t_train / 60, 1))
    print('\nRESULT:', row)
    json.dump(row, open(f'{run_dir}/result.json', 'w'), indent=1)
    append_result(row)
    shutil.rmtree(f'{run_dir}/ckpt', ignore_errors=True)
    return row

def run_or_load(name, enc, epochs=3, **kw):
    """Train `name` in this notebook run; if it already finished (e.g. before a disconnect), just load it."""
    d = f'{RUNS}/{name}'
    if os.path.exists(f'{d}/adapter/adapter_config.json') and os.path.exists(f'{d}/ep{epochs}/dev_score.json'):
        print(f'[{name}] already finished in this run folder -> loading')
        print_score(f'{name}/ep{epochs}', json.load(open(f'{d}/ep{epochs}/dev_score.json')))
        return json.load(open(f'{d}/result.json')) if os.path.exists(f'{d}/result.json') else None
    return run_experiment(name, enc, epochs=epochs, **kw)

In [10]:
# 0.10 Error-analysis helpers
def error_report(run, show=40):
    P = {p['id']: p['output'] for p in load_jsonl(run_file(run, 'dev_pred.jsonl'))}
    errs = []
    for i, g in gold.items():
        p = scorer.parse(P.get(i)) or {}
        wrong = [k for k in scorer.KEYS if not scorer.same(p.get(k), g[k])]
        if wrong:
            errs.append((i, wrong, {k: (g[k], p.get(k)) for k in wrong}))
    print(f'{run}: {len(errs)} of {len(gold)} dev rows have >=1 wrong field')
    print('wrong-field counts:', dict(collections.Counter(k for _, w, _ in errs for k in w)))
    print('by gold category  :', dict(collections.Counter(gold[i]['category'] for i, _, _ in errs)))
    for i, w, d in errs[:show]:
        print('=' * 100)
        print(i, '| gold category:', gold[i]['category'], '| (gold, pred):', d)
        print(dev_text[i].split('\n>')[0][:450])
    return errs

def confusion(run, field):
    P = {p['id']: scorer.parse(p['output']) or {} for p in load_jsonl(run_file(run, 'dev_pred.jsonl'))}
    return pd.crosstab(pd.Series([gold[i][field] for i in gold], name=f'gold {field}'),
                       pd.Series([str(P.get(i, {}).get(field)) for i in gold], name='pred'))
print('Setup complete.')

Setup complete.


## 1. Training data: rebuild the cleaned set from `train.jsonl`, then the augmentation (Parts A + D)

`data_prep.ipynb` is the audit: it shows how every problem was found and wrote `cleaning_log.csv`.
Here the cleaned set is **rebuilt from the raw file**: rows marked `drop` in the log are removed (truncated JSON,
placeholder tickets, dev leaks, exact/near duplicates), and every `fix` is **recomputed by code from the ticket text and
the SCHEMA rules** (legacy `prio` key, DD/MM dates, un-normalised ids, rupee amounts, priority by the v2 rules).
The set of rows the code changes must equal the log's `fix` rows, and every row must pass the validation gate.

In [11]:
# 1.1 The SCHEMA.md rules as code (identical to the audit)
def first_part(body):                      # the customer's own text: everything before quoted earlier emails
    return body.split('\n>')[0]

ENUMS = {
    'category': {'payment_failed', 'refund', 'fraud', 'account_access', 'kyc', 'offers', 'other'},
    'priority': {'P1', 'P2', 'P3'},
    'channel':  {None, 'upi', 'card', 'netbanking', 'wallet'},
    'language': {'en', 'hi-en'},
}
TXN_CATS = {'payment_failed', 'refund', 'fraud', 'offers'}
NON_TXN  = {'account_access', 'kyc', 'other'}
TXN_F    = ['amount_paise', 'txn_date', 'txn_id', 'channel']

def violations(y):
    v = []
    if list(y) != KEYS:
        v.append('keys')
    for k, allowed in ENUMS.items():
        if y.get(k) not in allowed:
            v.append(k)
    a = y.get('amount_paise')
    if a is not None and type(a) is not int:
        v.append('amount_type')
    d = y.get('txn_date')
    if d is not None and not (isinstance(d, str) and re.fullmatch(r'\d{4}-\d{2}-\d{2}', d)):
        v.append('date_format')
    t = y.get('txn_id')
    if t is not None and not (isinstance(t, str) and re.fullmatch(r'NP\d{10}', t)):
        v.append('txn_id_format')
    if type(y.get('needs_human')) is not bool:
        v.append('needs_human_type')
    return v

LEGAL = re.compile(r'\b(rbi|ombudsman|consumer\s+court|lawyer|legal\s+(notice|action))\b', re.I)
PRIOR_EXPLICIT = re.compile(
    r"(third|second|fourth|fifth|\d+(st|nd|rd|th)?) time I|again and again|"
    r"contacted (support|you) about this before|already (written|contacted|complained|raised|reported)|"
    r"written \d+ times|second complaint|keep raising|"
    r"baar baar|pehle bhi|teesri baar|dusri baar|doosri baar|kitni baar|phir se complaint", re.I)

def rule_priority(y, body):
    """SCHEMA.md v2 priority, applied in order (legal threats only in the customer's own text)."""
    c, a = y['category'], y['amount_paise']
    a = a if type(a) is int else None
    if c == 'fraud' or (a is not None and a >= 50_000_00) or LEGAL.search(first_part(body)):
        return 'P1'
    if (c in ('payment_failed', 'refund') and a is not None and a >= 5_000_00) or c == 'account_access':
        return 'P2'
    return 'P3'

NUM = r'\d[\d,]*(?:\.\d+)?'
MONEY_PATS = [
    re.compile(rf'(?:₹|\brs\.?|\binr)\s*(?P<n>{NUM})\s*(?P<m>k|lakhs?|lacs?)?\b', re.I),
    re.compile(rf'(?P<n>{NUM})\s*(?P<m>k|lakhs?|lacs?)\b', re.I),
    re.compile(rf'(?P<n>{NUM})\s*(?:rupees|rupee|rupaye|rupay)\b', re.I),
]
DISTRACT = re.compile(r'balance|limit|\bfees?\b|pade hain|bacha|remaining', re.I)

def to_paise(n, m=None):
    v = float(n.replace(',', ''))
    if m:
        v *= 1_000 if m.lower() == 'k' else 1_00_000
    return round(v * 100)

def boundaries(body):
    b = [-1]
    for m in re.finditer(r'[.!?](?=\s)|\n', body):
        if body[max(0, m.start() - 2):m.start()].lower() == 'rs':
            continue
        b.append(m.start())
    b.append(len(body))
    return b

def sentence_of(body, s, e):
    b = boundaries(body)
    return body[max(x for x in b if x < s) + 1:min(x for x in b if x >= e) + 1].strip()

def money_mentions2(body):
    out = []
    for p in MONEY_PATS:
        for mt in p.finditer(body):
            try:
                paise = to_paise(mt.group('n'), mt.groupdict().get('m'))
            except ValueError:
                continue
            if any(o['paise'] == paise and abs(o['start'] - mt.start()) < 12 for o in out):
                continue
            sent = sentence_of(body, mt.start(), mt.end())
            out.append(dict(paise=paise, start=mt.start(), sent=sent, distractor=bool(DISTRACT.search(sent))))
    return out

def amount_status2(y, body):
    a, ms = y['amount_paise'], money_mentions2(body)
    real = [m for m in ms if not m['distractor']]
    if a is None:
        if not ms:
            return 'null & no money'
        return 'null, only distractors' if not real else 'NULL BUT REAL AMOUNT'
    hit = [m for m in ms if m['paise'] == a]
    if hit:
        return 'match' if any(not m['distractor'] for m in hit) else 'TOOK DISTRACTOR'
    if any(a * 100 == m['paise'] or a == m['paise'] // 100 for m in ms):
        return 'RUPEES NOT CONVERTED'
    return 'NOT FOUND'

def check_rows(row_ids, lab, body):
    """Validation gate: format, priority rule, needs_human rule, null transaction fields, amount grounded in text."""
    problems = collections.Counter()
    for i in row_ids:
        y, b = lab[i], body[i]
        if violations(y):
            problems['format'] += 1
        if y['priority'] != rule_priority(y, b):
            problems['priority_rule'] += 1
        if y['needs_human'] != (y['priority'] == 'P1' or bool(PRIOR_EXPLICIT.search(first_part(b)))):
            problems['needs_human_rule'] += 1
        if y['category'] in NON_TXN and any(y[k] is not None for k in TXN_F):
            problems['txn_fields_not_null'] += 1
        if y['category'] in TXN_CATS and amount_status2(y, b) in ('NOT FOUND', 'RUPEES NOT CONVERTED', 'NULL BUT REAL AMOUNT'):
            problems['amount'] += 1
    return problems

HDR = re.compile(r'^Received: (\d{4}-\d{2}-\d{2}) \((\w{3})\)\nSource: ([\w-]+)\n\n')
def body_of_row(r):
    u = r['messages'][1]['content']
    return u[HDR.match(u).end():]

In [12]:
# 1.2 Rebuild train_clean from train.jsonl + cleaning_log.csv
clog = pd.read_csv(CLEAN_LOG)
DROP = set(clog.loc[clog['action'] == 'drop', 'id'])
FIX  = set(clog.loc[clog['action'] == 'fix', 'id'])
PRIO_MAP = {'high': 'P1', 'medium': 'P2', 'low': 'P3'}

def clean_label(y, body):
    """Apply the audit's provable fixes to one parsed label (cleaning_log.csv has the evidence for each)."""
    y = dict(y)
    if 'prio' in y:                                                     # legacy key low/medium/high (70 rows)
        y['priority'] = PRIO_MAP[y.pop('prio')]
    d = y.get('txn_date')
    if isinstance(d, str) and re.fullmatch(r'\d{2}/\d{2}/\d{4}', d):    # DD/MM/YYYY (30 rows)
        dd, mm, yy = d.split('/')
        y['txn_date'] = f'{yy}-{mm}-{dd}'
    t = y.get('txn_id')
    if isinstance(t, str) and not re.fullmatch(r'NP\d{10}', t):        # un-normalised id (20 rows)
        y['txn_id'] = re.sub(r'[^A-Za-z0-9]', '', t).upper()
    y = {k: y[k] for k in KEYS}
    if amount_status2(y, body) == 'RUPEES NOT CONVERTED':              # rupees stored as paise (25 rows)
        a = y['amount_paise']
        cands = {m['paise'] for m in money_mentions2(body) if a * 100 == m['paise'] or a == m['paise'] // 100}
        assert len(cands) == 1, cands
        y['amount_paise'] = cands.pop()
    y['priority'] = rule_priority(y, body)                              # SCHEMA v2 priority (60 rows)
    return y

clean_rows, changed, LAB_C, BODY_C = [], set(), {}, {}
for r in train_raw:
    if r['id'] in DROP:
        continue
    b = body_of_row(r)
    y0 = json.loads(r['messages'][2]['content'])
    y = clean_label(y0, b)
    if y != y0:
        changed.add(r['id'])
    LAB_C[r['id']], BODY_C[r['id']] = y, b
    clean_rows.append({'id': r['id'], 'messages': [r['messages'][0], r['messages'][1],
                       {'role': 'assistant', 'content': json.dumps(y, ensure_ascii=False)}]})

print(f'raw {len(train_raw)} - dropped {len(DROP)} = {len(clean_rows)} rows | rows changed by code: {len(changed)} '
      f'| log says fixed: {len(FIX)}')
print('changed by code but not in log:', sorted(changed - FIX)[:10], '| in log but unchanged:', sorted(FIX - changed)[:10])
gate = check_rows(list(LAB_C), LAB_C, BODY_C)
print('validation gate:', dict(gate) or 'NONE')
REBUILD_OK = len(clean_rows) == 1262 and changed == FIX and not gate

if os.path.exists(REF_CLEAN):                                          # prove it equals data_prep.ipynb's output
    ref = [json.dumps(r, ensure_ascii=False) for r in load_jsonl(REF_CLEAN)]
    mine = [json.dumps(r, ensure_ascii=False) for r in clean_rows]
    same = ref == mine
    print('identical to data_prep.ipynb train_clean.jsonl:', same)
    if not same:
        print('WARNING: rebuild differs from the reference -> using the reference file for training')
        clean_rows = load_jsonl(REF_CLEAN)
    REBUILD_OK = REBUILD_OK and same
assert len(clean_rows) == 1262
print('REBUILD_OK =', REBUILD_OK)
train = clean_rows
os.makedirs(OUT, exist_ok=True)
with open(f'{OUT}/train_clean.jsonl', 'w', encoding='utf-8') as f:
    for r in train:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

raw 1445 - dropped 183 = 1262 rows | rows changed by code: 175 | log says fixed: 175
changed by code but not in log: [] | in log but unchanged: []
validation gate: NONE
identical to data_prep.ipynb train_clean.jsonl: True
REBUILD_OK = True


### 1.3 Rule-boundary augmentation (experiment D4, the final model's training data)

R1's dev errors were mostly **priority decisions on correctly extracted values**: amounts near the ₹50k / ₹5k
thresholds (wrong side, both directions) and a missed legal threat. Train has few tickets near those boundaries.
So: take real cleaned tickets and change exactly one decision-relevant thing (**amt**: the amount, to a value near
₹5k/₹50k in the same written style; **legal**: append a legal-threat sentence harvested from train), and recompute the
label with the SCHEMA rules. Only `train.jsonl` and `SCHEMA.md` are used; every new row passes the same gate.

In [13]:
# 1.3 Augmentation (seeded: identical every run)
def make_augmentation(clean_rows, seed=1234, n_amt=250, n_legal=120):
    rng = random.Random(seed)
    keep_ids = [r['id'] for r in clean_rows]
    user_of  = {r['id']: r['messages'][1]['content'] for r in clean_rows}
    body_of  = {r['id']: body_of_row(r) for r in clean_rows}
    LAB      = {r['id']: json.loads(r['messages'][2]['content']) for r in clean_rows}
    sys_prompt = clean_rows[0]['messages'][0]['content']
    aug, aug_log, aug_lab, aug_body = [], [], {}, {}

    def indian(n):                                     # 168787 -> '1,68,787'
        s = str(n)
        if len(s) <= 3:
            return s
        head, tail, parts = s[:-3], s[-3:], []
        while len(head) > 2:
            parts.insert(0, head[-2:]); head = head[:-2]
        return ','.join(([head] if head else []) + parts + [tail])

    def find_amount_match(body, paise):
        hits = []
        for p in MONEY_PATS:
            for mt in p.finditer(body):
                try:
                    v = to_paise(mt.group('n'), mt.groupdict().get('m'))
                except ValueError:
                    continue
                if v == paise and not DISTRACT.search(sentence_of(body, mt.start(), mt.end())):
                    hits.append(mt)
        spans = {mt.span('n') for mt in hits}
        return sorted(hits, key=lambda mt: mt.start('n'))[0] if len(spans) == 1 else None

    def sample_rupees(T):
        if rng.random() < 0.10:
            return T
        return int(round(math.exp(rng.uniform(math.log(0.4 * T), math.log(2.4 * T))), -2))

    def render(old_n, mult, rupees):
        if mult and mult.lower() == 'k':
            return f'{rupees/1000:.1f}'.rstrip('0').rstrip('.')
        s = indian(rupees) if ',' in old_n else str(rupees)
        return s + (f'.{rng.randint(1, 99):02d}' if '.' in old_n else '')

    def relabel(y, new_body, **changes):
        y2 = dict(y); y2.update(changes)
        y2['priority'] = rule_priority(y2, new_body)
        y2['needs_human'] = y2['priority'] == 'P1' or bool(PRIOR_EXPLICIT.search(first_part(new_body)))
        return {k: y2[k] for k in KEYS}

    def add_aug(src, kind, new_body, y_new, extra):
        aid = f'aug-{kind}-{len(aug) + 1:04d}'
        user = user_of[src][:len(user_of[src]) - len(body_of[src])] + new_body
        aug.append({'id': aid, 'messages': [
            {'role': 'system', 'content': sys_prompt}, {'role': 'user', 'content': user},
            {'role': 'assistant', 'content': json.dumps(y_new, ensure_ascii=False)}]})
        aug_lab[aid], aug_body[aid] = y_new, new_body
        aug_log.append({'id': aid, 'source': src, 'kind': kind, **extra})

    # amount-boundary examples
    amt_cands = []
    for i in keep_ids:
        y, b = LAB[i], body_of[i]
        if y['category'] not in ('payment_failed', 'refund') or y['amount_paise'] is None:
            continue
        if LEGAL.search(first_part(b)):
            continue
        mt = find_amount_match(b, y['amount_paise'])
        if mt is None or (mt.groupdict().get('m') or '').lower().startswith('l'):
            continue
        amt_cands.append((i, mt))
    for i, mt in rng.sample(amt_cands, min(n_amt, len(amt_cands))):
        b, y = body_of[i], LAB[i]
        mult = mt.groupdict().get('m')
        new_n = render(mt.group('n'), mult, sample_rupees(rng.choice([5_000, 50_000])))
        s, e = mt.span('n')
        nb = b[:s] + new_n + b[e:]
        y2 = relabel(y, nb, amount_paise=to_paise(new_n, mult))
        assert amount_status2(y2, nb) == 'match', (i, new_n)
        add_aug(i, 'amt', nb, y2, {'old_amt': y['amount_paise'], 'new_amt': y2['amount_paise'],
                                   'pri': f"{y['priority']}->{y2['priority']}"})

    # legal-threat examples
    greetings = ('hi', 'hello', 'hey', 'dear', 'namaste', 'bhai', 'sir', 'good morning', '@')
    legal_sents = {'en': set(), 'hi-en': set()}
    for i in keep_ids:
        fp = first_part(body_of[i])
        m = LEGAL.search(fp)
        if m:
            s = sentence_of(fp, m.start(), m.end())
            if not re.search(r'\d', s) and len(s) < 80 and ',' not in s and not s.lower().startswith(greetings):
                legal_sents[LAB[i]['language']].add(s)
    legal_sents = {k: sorted(v) for k, v in legal_sents.items()}

    def insert_sentence(body, sent):
        cut = len(body)
        for marker in ('\n\nRegards', '\n\n>'):
            k = body.find(marker)
            if k != -1:
                cut = min(cut, k)
        head = body[:cut].rstrip()
        return head + (' ' if head[-1:] in '.?!' else '. ') + sent + body[cut:]

    leg_cands = [i for i in keep_ids if LAB[i]['category'] != 'fraud' and LAB[i]['priority'] != 'P1'
                 and not LEGAL.search(first_part(body_of[i]))]
    for i in rng.sample(leg_cands, min(n_legal, len(leg_cands))):
        y = LAB[i]
        nb = insert_sentence(body_of[i], rng.choice(legal_sents[y['language']] or legal_sents['en']))
        y2 = relabel(y, nb)
        assert y2['priority'] == 'P1' and y2['needs_human']
        add_aug(i, 'legal', nb, y2, {'pri': f"{y['priority']}->P1"})
    return aug, pd.DataFrame(aug_log), aug_lab, aug_body, legal_sents

aug, aug_log, aug_lab, aug_body, legal_sents = make_augmentation(train)
gate = check_rows(list(aug_lab), aug_lab, aug_body)
print('augmented:', aug_log['kind'].value_counts().to_dict(), '| validation gate:', dict(gate) or 'NONE')
print('legal sentences used:', legal_sents)
print('priority transitions:\n', aug_log.groupby('kind')['pri'].value_counts().to_string())
train_aug = train + aug
AUG_OK = not gate and len(aug) == 370
if os.path.exists(REF_AUG):
    same = [json.dumps(r, ensure_ascii=False) for r in load_jsonl(REF_AUG)] == [json.dumps(r, ensure_ascii=False) for r in train_aug]
    print('identical to exp4_augmentation.ipynb train_aug.jsonl:', same)
    if not same:
        print('WARNING: augmentation differs from the reference -> using the reference file for training')
        train_aug = load_jsonl(REF_AUG)
    AUG_OK = AUG_OK and same
print('AUG_OK =', AUG_OK, '| train_aug rows:', len(train_aug))
with open(f'{OUT}/train_aug.jsonl', 'w', encoding='utf-8') as f:
    for r in train_aug:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
aug_log.to_csv(f'{OUT}/augmentation_log.csv', index=False)

augmented: {'amt': 250, 'legal': 120} | validation gate: NONE
legal sentences used: {'en': ['I will complain to the RBI ombudsman.', 'I will escalate this to the RBI.', 'I will go to consumer court.', 'My lawyer will send a legal notice.', 'Otherwise I will approach the consumer court.'], 'hi-en': ['Consumer court jaunga.', 'RBI mein complaint kar dunga.']}
priority transitions:
 kind   pri   
amt    P3->P2    82
       P3->P1    38
       P2->P2    34
       P3->P3    29
       P2->P1    25
       P2->P3    15
       P1->P2    13
       P1->P1    11
       P1->P3     3
legal  P3->P1    82
       P2->P1    38
identical to exp4_augmentation.ipynb train_aug.jsonl: True
AUG_OK = True | train_aug rows: 1632


In [14]:
# 1.4 Encode (verifies on every row that our tokens equal the chat template's)
enc_clean, enc_aug = encode_all(train), encode_all(train_aug)
print('encoded:', len(enc_clean), 'clean |', len(enc_aug), 'clean+augmented')

encoded: 1262 clean | 1632 clean+augmented


## 2. Chat template and token lengths (Part C evidence)

ChatML; the prompt ends with `<|im_start|>assistant\n`; the answer ends with `<|im_end|>` (= eos, kept in the loss);
pad ≠ eos. Without a system message Qwen injects "You are Qwen…", so the fixed one is always passed. The default
generation config has `repetition_penalty=1.05`, which stays active under greedy decoding (measured: it changes 2/200
dev outputs and not the score). Answers ≤ 85 tokens; 97% of rows ≤ 256 tokens, 36 quoted emails up to 2,215 →
`MAX_LEN = 2304`, no truncation (truncating to 512 would delete those rows' entire labels).

In [15]:
ex = train[0]['messages']
prompt, full = build_prompt(ex), tok.apply_chat_template(ex, tokenize=False)
print('=== PROMPT ===\n' + prompt)
print('=== after the prompt ===', repr(full[len(prompt):]))
p_ids, f_ids = ids(prompt), ids(full)
print('prompt is a token prefix of the full sequence:', f_ids[:len(p_ids)] == p_ids)
print('=== without a system message ===\n' + tok.apply_chat_template(ex[1:2], tokenize=False, add_generation_prompt=True))
print('=== default generation config ===\n', GenerationConfig.from_pretrained(MODEL_ID))

def token_lengths(rows):
    out = []
    for r in rows:
        p = len(ids(build_prompt(r['messages'])))
        t = len(ids(tok.apply_chat_template(r['messages'], tokenize=False))) if len(r['messages']) == 3 else p
        out.append((r['id'], p, t - p, t))
    return pd.DataFrame(out, columns=['id', 'prompt', 'answer', 'total'])

for name, rows in [('train_clean', train), ('dev', dev), ('test (prompt only)', test)]:
    df = token_lengths(rows)
    print(f'\n--- {name} ({len(df)} rows)')
    print(df[['prompt', 'answer', 'total']].quantile([.5, .9, .99, 1.0]).astype(int).T
            .rename(columns={.5: 'p50', .9: 'p90', .99: 'p99', 1.0: 'max'}))
    print('   rows with total >', {t: int((df.total > t).sum()) for t in [256, 512, 1024, 2048]})

=== PROMPT ===
<|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-07-25 (Sat)
Source: app-chat

Hi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply.<|im_end|>
<|im_start|>assistant

=== after the prompt === '{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>\n'
prompt is a token prefix of the full sequence: True
=== without a system message ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Received: 2026-07-25 (Sat)
Source: app-chat

Hi, I have a top up that didn't go through. Why? It was today. Mode: 

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

=== default generation config ===
 GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}


--- train_clean (1262 rows)
        p50  p90   p99   max
prompt  103  135  1784  2145
answer   66   80    83    85
total   168  214  1862  2215
   rows with total > {256: 39, 512: 36, 1024: 36, 2048: 9}

--- dev (200 rows)
        p50  p90   p99   max
prompt  106  136  1464  2115
answer   68   81    83    83
total   173  213  1516  2181
   rows with total > {256: 3, 512: 3, 1024: 3, 2048: 2}

--- test (prompt only) (400 rows)
        p50  p90   p99   max
prompt  108  143  1827  2089
answer    0    0     0     0
total   108  143  1827  2089
   rows with total > {256: 11, 512: 11, 1024: 11, 2048: 1}


## 3. Baselines (Part B), shown from the saved logs

B1: base model, fixed prompt. B2: base model with SCHEMA.md in the system prompt (allowed only here, as a reference).
B1 wraps its output in fences and invents its own schema (0%); B2 produces valid JSON but applies the multi-step rules
poorly (priority 51%) and copies SCHEMA.md's example id into 13/200 answers. The code that produced them is below.

In [16]:
SCHEMA_MD = open(f'{ROOT}/candidate_pack/SCHEMA.md', encoding='utf-8').read()
SYS_RULES = (SYSTEM + '\n\nApply the following labelling specification exactly. '
             'Respond with only the JSON object and nothing else.\n\n' + SCHEMA_MD)
RERUN_BASELINES = False                            # True: regenerate (GPU, ~8 min) instead of loading the saved logs
if RERUN_BASELINES:
    model = load_for_inference()
    for name, system in [('B1_zeroshot', None), ('B2_rules_in_prompt', SYS_RULES)]:
        evaluate(generate_outputs(dev, model, system=system), name)
    free_gpu()
for name in ['B1_zeroshot', 'B2_rules_in_prompt']:
    print_score(name, saved_score(name))
pred_b2 = load_jsonl(run_file('B2_rules_in_prompt', 'dev_pred.jsonl'))
print('B2 answers containing SCHEMA.md\'s example id NP1234567890:', sum('NP1234567890' in p['output'] for p in pred_b2))
display(confusion('B2_rules_in_prompt', 'priority'))

  [B1_zeroshot] valid 0.5% | exact 0.0% | field 0.0% | catego=0% priori=0% amount=0% txn_da=0% txn_id=0% channe=0% langua=0% needs_=0%
  [B2_rules_in_prompt] valid 100.0% | exact 23.5% | field 80.6% | catego=84% priori=51% amount=86% txn_da=83% txn_id=89% channe=85% langua=90% needs_=76%
B2 answers containing SCHEMA.md's example id NP1234567890: 13


pred,P1,P2,P3
gold priority,,,
P1,18,21,3
P2,3,32,17
P3,4,50,52


## 4. Pipeline checks (Part C)

In [17]:
e = enc_clean[0]
n_lab = sum(l != -100 for l in e['labels'])
print('LAST PROMPT TOKENS (no loss):', repr(tok.decode(e['input_ids'][-n_lab - 15:-n_lab])))
print('TOKENS WITH LOSS            :', repr(tok.decode(e['input_ids'][-n_lab:])))
tot = sum(len(e['input_ids']) for e in enc_clean)
lab_ = sum(sum(l != -100 for l in e['labels']) for e in enc_clean)
print(f'share of tokens with loss: {lab_/tot:.1%}')
rand = [list(range(len(enc_clean)))[i:i + 4] for i in range(0, len(enc_clean), 4)]
print(f'padding efficiency: file order {padding_efficiency(enc_clean, rand):.1%} -> '
      f'length-grouped {padding_efficiency(enc_clean, length_grouped_batches(enc_clean, 4, 0)):.1%}')

base = load_base(quant4=False)
pm = add_lora(base, r=16)
worst = sorted(range(len(enc_clean)), key=lambda k: -len(enc_clean[k]['input_ids']))[:4]
pm.train(); torch.cuda.reset_peak_memory_stats()
with torch.autocast('cuda', dtype=torch.float16):
    loss = answer_loss(pm, collate(enc_clean, worst))
loss.backward(); torch.cuda.synchronize()
print(f'worst-case batch {[len(enc_clean[k]["input_ids"]) for k in worst]}: '
      f'PEAK {torch.cuda.max_memory_allocated()/2**30:.2f} GiB of 15')
del loss
free_gpu()

LAST PROMPT TOKENS (no loss): '6332. Waiting for your reply.<|im_end|>\n<|im_start|>assistant\n'
TOKENS WITH LOSS            : '{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>'
share of tokens with loss: 29.8%
padding efficiency: file order 58.7% -> length-grouped 80.2%
allocated 0.00 GiB | reserved 0.00 GiB


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29.93M (cap 50M) | dtypes: {torch.float32}
worst-case batch [2214, 2210, 2175, 2168]: PEAK 9.62 GiB of 15
allocated 0.02 GiB | reserved 5.75 GiB


## 5. Controlled experiments (Part D), shown from the saved logs

Each run changes **one** thing versus the reference R1 (clean data, fp16 LoRA r=16, α=32, lr 2e-4, 5% warmup + cosine,
effective batch 16, 3 epochs, loss on answer tokens only). With 200 dev rows, 1 row = 0.5 points (SE ≈ ±2 points), so
each run is also compared **paired** with R1 (rows fixed vs broken, exact McNemar test).
The code that trained them is the cell below (set `RERUN_EXPERIMENTS = True` to retrain; ~30–40 min each).

In [18]:
RERUN_EXPERIMENTS = False
EXPERIMENTS = {
    R1:                    dict(enc='clean', note='reference: clean data, 16-bit LoRA r16, lr 2e-4, 3 epochs, eff. batch 16'),
    'D1_raw_lora16_fp16':  dict(enc='raw',   note='Exp1: raw train.jsonl as given'),
    'D2_clean_qlora_4bit': dict(enc='clean', quant4=True, note='Exp2: 4-bit NF4 QLoRA'),
    'D3_clean_lora8_fp16': dict(enc='clean', r=8, note='Exp3: LoRA rank 8 (alpha 16)'),
    D4:                    dict(enc='aug',   note='Exp4: clean + rule-boundary augmentation'),
}
if RERUN_EXPERIMENTS:                      # restart the session between runs for comparable peak memory
    enc_raw = encode_all(train_raw)
    for name, cfg in EXPERIMENTS.items():
        cfg = dict(cfg)
        enc = {'clean': enc_clean, 'raw': enc_raw, 'aug': enc_aug}[cfg.pop('enc')]
        run_or_load(name, enc, **cfg)
else:
    print('experiments shown from saved logs:', LOGS)

experiments shown from saved logs: /content/drive/MyDrive/vima3ya/runs


In [19]:
WHAT = {
    'B1_zeroshot':          'baseline: base model, fixed prompt',
    'B2_rules_in_prompt':   'baseline: base model + rules in prompt',
    R1:                     'reference: clean, 16-bit LoRA r16',
    'D1_raw_lora16_fp16':   'Exp1: raw data instead of clean',
    'D2_clean_qlora_4bit':  'Exp2: 4-bit QLoRA instead of 16-bit',
    'D3_clean_lora8_fp16':  'Exp3: rank 8 instead of 16',
    D4:                     'Exp4: + rule-boundary augmentation',
}
res = pd.read_csv(f'{LOGS}/results.csv')
for p in sorted(glob.glob(f'{LOGS}/*/result.json')):                # rows saved per run (multi-account runs)
    row = json.load(open(p))
    res = pd.concat([res[res['run'] != row['run']], pd.DataFrame([row])], ignore_index=True)
res = res[res['run'].isin(WHAT)].copy()
res['what changed'] = res['run'].map(WHAT)
res['order'] = res['run'].map({k: i for i, k in enumerate(WHAT)})
tbl = res.sort_values('order')[['run', 'what changed', 'exact', 'mean_field', 'json_valid', 'peak_train_gib', 'train_min', 'n_train']].copy()
for c in ['exact', 'mean_field', 'json_valid']:
    tbl[c] = (tbl[c] * 100).round(1)
tbl.loc[tbl['run'] == R1, 'peak_train_gib'] = 9.96       # R1's own number (10.36) included 0.42 GiB left over from earlier cells
display(tbl.rename(columns={'exact': 'dev exact %', 'mean_field': 'dev field %', 'json_valid': 'valid JSON %',
                            'peak_train_gib': 'peak GPU GiB*', 'train_min': 'train min'}))
print('* training peak; B1/B2: inference peak\n')
for name in ['D1_raw_lora16_fp16', 'D2_clean_qlora_4bit', 'D3_clean_lora8_fp16', D4]:
    paired(f'{R1}/ep3', f'{name}/ep3')
print('\nper-epoch (R1 and D4):')
paired(f'{R1}/ep1', f'{R1}/ep2'); paired(f'{R1}/ep2', f'{R1}/ep3'); paired(f'{R1}/ep3', f'{D4}/ep2')

,run,what changed,dev exact %,dev field %,valid JSON %,peak GPU GiB*,train min,n_train
1,B1_zeroshot,"baseline: base model, fixed prompt",0.0,0.0,0.5,7.68,0.0,0
2,B2_rules_in_prompt,baseline: base model + rules in prompt,23.5,80.6,100.0,8.93,0.0,0
0,R1_clean_lora16_fp16,"reference: clean, 16-bit LoRA r16",91.5,98.2,100.0,9.96,27.5,1262
3,D1_raw_lora16_fp16,Exp1: raw data instead of clean,92.5,98.5,100.0,9.96,30.1,1445
4,D2_clean_qlora_4bit,Exp2: 4-bit QLoRA instead of 16-bit,93.5,98.7,100.0,8.29,34.3,1262
5,D3_clean_lora8_fp16,Exp3: rank 8 instead of 16,90.0,98.1,100.0,9.73,27.0,1262
6,D4_aug_lora16_fp16,Exp4: + rule-boundary augmentation,95.0,99.0,100.0,9.96,38.1,1632


* training peak; B1/B2: inference peak

R1_clean_lora16_fp16/ep3  ->  D1_raw_lora16_fp16/ep3: exact 183/200 -> 185/200 | fixed 5, broke 3 | McNemar p = 0.727
R1_clean_lora16_fp16/ep3  ->  D2_clean_qlora_4bit/ep3: exact 183/200 -> 187/200 | fixed 8, broke 4 | McNemar p = 0.388
R1_clean_lora16_fp16/ep3  ->  D3_clean_lora8_fp16/ep3: exact 183/200 -> 180/200 | fixed 2, broke 5 | McNemar p = 0.453
R1_clean_lora16_fp16/ep3  ->  D4_aug_lora16_fp16/ep3: exact 183/200 -> 190/200 | fixed 8, broke 1 | McNemar p = 0.039

per-epoch (R1 and D4):
R1_clean_lora16_fp16/ep1  ->  R1_clean_lora16_fp16/ep2: exact 159/200 -> 178/200 | fixed 23, broke 4 | McNemar p = 0.000
R1_clean_lora16_fp16/ep2  ->  R1_clean_lora16_fp16/ep3: exact 178/200 -> 183/200 | fixed 6, broke 1 | McNemar p = 0.125
R1_clean_lora16_fp16/ep3  ->  D4_aug_lora16_fp16/ep2: exact 183/200 -> 188/200 | fixed 9, broke 4 | McNemar p = 0.267


In [20]:
# Exp 1 made fair: raw train contains copies/variants of 47 dev tickets -> also compare on the leak-free dev subset
leak_dev = set(re.findall(r'dv-\d{5}', ' '.join(clog['reason'].astype(str))))
_body = lambda s: re.sub(r'\s+', ' ', s.split('\n\n', 1)[-1]).strip().lower()
raw_bodies = {_body(r['messages'][1]['content']) for r in train_raw}
leak_dev |= {r['id'] for r in dev if _body(r['messages'][1]['content']) in raw_bodies}
clean_ids = set(gold) - leak_dev
print(f'dev tickets with a copy/variant in raw train: {len(leak_dev)} -> leak-free subset: {len(clean_ids)}')
for name in [R1, 'D1_raw_lora16_fp16']:
    c, l = score_subset(f'{name}/ep3', clean_ids), score_subset(f'{name}/ep3', leak_dev)
    print(f'{name:22} leak-free exact {c["exact_match"]:.1%} | leaked rows only {l["exact_match"]:.1%}')
paired(f'{R1}/ep3', 'D1_raw_lora16_fp16/ep3', ids=clean_ids)

dev tickets with a copy/variant in raw train: 47 -> leak-free subset: 153
R1_clean_lora16_fp16   leak-free exact 89.5% | leaked rows only 97.9%
D1_raw_lora16_fp16     leak-free exact 90.2% | leaked rows only 100.0%
R1_clean_lora16_fp16/ep3  ->  D1_raw_lora16_fp16/ep3: exact 137/153 -> 138/153 | fixed 4, broke 3 | McNemar p = 1.000


## 6. Train the final model (D4 configuration, from scratch)

Same configuration as D4. GPU kernels are not bit-deterministic, so identical runs differ by a few dev rows
(D4's first run 95.0%, an identical retrain 93.5%); the paired tests below put that in context.

In [21]:
FINAL = 'FINAL_aug_lora16_fp16'
FINAL_CFG = dict(quant4=False, r=16, alpha=32, lr=2e-4, epochs=3, bs=4, accum=4, seed=0, warmup_frac=0.05)
if LOGS and os.path.exists(f'{LOGS}/{D4}/config.json'):
    c = json.load(open(f'{LOGS}/{D4}/config.json'))
    assert all(c[k] == v for k, v in FINAL_CFG.items()), 'FINAL_CFG differs from D4'
final_row = run_or_load(FINAL, enc_aug, note='final model: D4 configuration retrained by this notebook', **FINAL_CFG)
free_gpu()
paired(f'{R1}/ep3', f'{FINAL}/ep3')
paired(f'{D4}/ep3', f'{FINAL}/ep3')

allocated 0.02 GiB | reserved 5.75 GiB


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29.93M (cap 50M) | dtypes: {torch.float32}
  step    1/306 | ep 1 | loss 2.5548 | lr 2.67e-05 | gnorm 7.06 | scale 65536 | 0.1 min
  step   10/306 | ep 1 | loss 0.4766 | lr 1.47e-04 | gnorm 1.98 | scale 65536 | 1.2 min
  step   20/306 | ep 1 | loss 0.0796 | lr 2.00e-04 | gnorm 0.61 | scale 65536 | 2.1 min
  [ckpt] saved step 25 (epoch 0)
  step   30/306 | ep 1 | loss 0.0399 | lr 1.99e-04 | gnorm 0.36 | scale 65536 | 3.3 min
  step   40/306 | ep 1 | loss 0.0209 | lr 1.96e-04 | gnorm 0.13 | scale 65536 | 4.4 min
  step   50/306 | ep 1 | loss 0.0338 | lr 1.93e-04 | gnorm 0.19 | scale 65536 | 5.4 min
  [ckpt] saved step 50 (epoch 0)
  step   60/306 | ep 1 | loss 0.0116 | lr 1.88e-04 | gnorm 0.14 | scale 65536 | 6.8 min
  step   70/306 | ep 1 | loss 0.0223 | lr 1.83e-04 | gnorm 0.19 | scale 65536 | 7.5 min
  [ckpt] saved step 75 (epoch 0)
  step   80/306 | ep 1 | loss 0.0117 | lr 1.76e-04 | gnorm 0.17 | scale 65536 | 8.4 min
  step   90/306 | ep 1 | loss 0.0018 | lr 1.69e-

## 7. Error analysis of the final model on dev (Part E)

In [22]:
errs = error_report(f'{FINAL}/ep3')
print('\npriority confusion:'); display(confusion(f'{FINAL}/ep3', 'priority'))
print('category confusion:'); display(confusion(f'{FINAL}/ep3', 'category'))
meta = pd.DataFrame({
    'exact':    row_exact(f'{FINAL}/ep3'),
    'category': {i: gold[i]['category'] for i in gold},
    'source':   {r['id']: re.search(r'Source: ([\w-]+)', r['messages'][1]['content']).group(1) for r in dev},
    'language': {i: gold[i]['language'] for i in gold},
    'quoted':   {r['id']: bool(re.search(r'^>', r['messages'][1]['content'], re.M)) for r in dev},
})
for c in ['category', 'source', 'language', 'quoted']:
    print(f'\nexact match by {c}:\n', meta.groupby(c)['exact'].agg(['mean', 'size']).round(3).to_string())

FINAL_aug_lora16_fp16/ep3: 12 of 200 dev rows have >=1 wrong field
wrong-field counts: {'category': 3, 'priority': 7, 'needs_human': 5, 'txn_date': 3, 'txn_id': 1}
by gold category  : {'payment_failed': 4, 'refund': 7, 'fraud': 1}
dv-00010 | gold category: payment_failed | (gold, pred): {'category': ('payment_failed', 'fraud'), 'priority': ('P3', 'P1'), 'needs_human': (False, True)}
Received: 2026-07-09 (Thu)
Source: email

Subject: Problem with NimbusPay

URGENT!!
I didn't see the money I added to my card. ID: NP0194178721 INR 2,570 was the value. Please look into it.

Regards,
Vikram Singh
dv-00037 | gold category: refund | (gold, pred): {'priority': ('P1', 'P2'), 'needs_human': (True, False)}
Received: 2026-09-07 (Mon)
Source: app-chat

My transaction history is looking a little off. I have been charged twice for multiple different purchases, and would like to have those duplicate charges refunded. The txn id is np 42846 07517. Transaction date 6 Sep. 284669 rupees was the value. Wa

pred,P1,P2,P3
gold priority,,,
P1,40,1,1
P2,2,50,0
P3,2,1,103


category confusion:


pred,account_access,fraud,kyc,offers,other,payment_failed,refund
gold category,,,,,,,
account_access,19,0,0,0,0,0,0
fraud,1,14,0,0,0,0,0
kyc,0,0,24,0,0,0,0
offers,0,0,0,18,0,0,0
other,0,0,0,0,16,0,0
payment_failed,0,2,0,0,0,51,0
refund,0,0,0,0,0,0,55



exact match by category:
                  mean  size
category                   
account_access  1.000    19
fraud           0.933    15
kyc             1.000    24
offers          1.000    18
other           1.000    16
payment_failed  0.925    53
refund          0.873    55

exact match by source:
                  mean  size
source                     
app-chat        0.907    86
email           0.977    44
ivr-transcript  0.933    30
twitter         0.975    40

exact match by language:
            mean  size
language             
en        0.947   151
hi-en     0.918    49

exact match by quoted:
          mean  size
quoted             
False   0.939   197
True    1.000     3


## 8. Test predictions (one row at a time, exactly as the graders run the model) + checks

The adapter is exported in fp16 (~57 MB), reloaded as the graders will (fp16 base + PEFT adapter, **not 4-bit**), and
each test row is generated with `model.generate(**inputs, max_new_tokens=200, do_sample=False)` using the model's own
chat template and default generation config; the only post-processing is `.strip()`.

In [23]:
export_fp16_adapter(f'{RUNS}/{FINAL}/adapter', f'{OUT}/adapter')
model = load_for_inference(f'{OUT}/adapter')
PRED = f'{OUT}/predictions_test.jsonl'
done = {p['id']: p['output'] for p in load_jsonl(PRED)} if os.path.exists(PRED) else {}
print('already predicted (resume):', len(done))
t0 = time.time()
with open(PRED, 'a', encoding='utf-8') as f:
    for k, r in enumerate(test, 1):
        if r['id'] in done:
            continue
        done[r['id']] = predict_one(model, r)
        f.write(json.dumps({'id': r['id'], 'output': done[r['id']]}, ensure_ascii=False) + '\n'); f.flush()
        if k % 50 == 0:
            print(f'  test {k}/{len(test)} | {(time.time()-t0)/60:.1f} min')
with open(PRED, 'w', encoding='utf-8') as f:                       # final file: one line per test row, in test order
    for r in test:
        f.write(json.dumps({'id': r['id'], 'output': done[r['id']]}, ensure_ascii=False) + '\n')
preds = load_jsonl(PRED)
print('predictions_test.jsonl:', len(preds), 'rows | valid JSON:', sum(scorer.parse(p['output']) is not None for p in preds))

adapter exported in fp16: 57 MB -> /content/drive/MyDrive/vima3ya/official/adapter
allocated 0.02 GiB | reserved 5.75 GiB


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

already predicted (resume): 0
  test 50/400 | 5.8 min
  test 100/400 | 11.6 min
  test 150/400 | 17.4 min
  test 200/400 | 23.1 min
  test 250/400 | 28.8 min
  test 300/400 | 35.2 min
  test 350/400 | 40.7 min
  test 400/400 | 46.6 min
predictions_test.jsonl: 400 rows | valid JSON: 400


In [24]:
# Reproducibility: fresh reload, regenerate 20 random test rows, compare with the file (what the graders do)
model = load_for_inference(f'{OUT}/adapter')
saved = {p['id']: p['output'] for p in load_jsonl(PRED)}
sample = random.Random(0).sample(test, 20)
print(f'reproducibility: {sum(predict_one(model, r) == saved[r["id"]] for r in sample)}/20 identical')
# Batched training-time dev predictions vs one-row-at-a-time with the exported fp16 adapter (first 40 dev rows)
batched = {p['id']: p['output'] for p in load_jsonl(run_file(f'{FINAL}/ep3', 'dev_pred.jsonl'))}
print(f"dev batched vs one-at-a-time identical: {sum(predict_one(model, r) == batched[r['id']] for r in dev[:40])}/40")
free_gpu()

allocated 0.02 GiB | reserved 5.75 GiB


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

reproducibility: 20/20 identical
dev batched vs one-at-a-time identical: 40/40
allocated 0.02 GiB | reserved 5.75 GiB


In [25]:
# Label-free sanity check of the test predictions (diagnostic only: the file is not modified)
tp   = {p['id']: json.loads(p['output']) for p in preds}
text = {r['id']: r['messages'][1]['content'] for r in test}
recv = {i: re.search(r'Received: (\d{4}-\d{2}-\d{2})', t).group(1) for i, t in text.items()}
body = {i: first_part(t.split('\n\n', 1)[-1]) for i, t in text.items()}
chk = collections.Counter()
for i, y in tp.items():
    chk['exactly the 8 keys, in order'] += list(y) == KEYS
    chk['priority = SCHEMA rule on its own fields'] += y['priority'] == rule_priority(y, body[i])
    chk['txn fields null for account_access/kyc/other'] += y['category'] not in NON_TXN or all(y[k] is None for k in TXN_F)
    chk['txn_id is NP + 10 digits (or null)'] += y['txn_id'] is None or bool(re.fullmatch(r'NP\d{10}', str(y['txn_id'])))
    chk['txn_date not after Received'] += y['txn_date'] is None or y['txn_date'] <= recv[i]
for k, v in chk.items():
    print(f'{k:48} {v}/{len(tp)}')

# Phrasings in test that never occur in train or dev (the "harder" slice)
splits = {'train': train_raw, 'dev': dev, 'test': test}
pats = {'order number': r'order (number|no)', 'convenience fee': r'convenience fee', '"I am out X"': r'\bI am out\b',
        '"sum involved"': r'sum involved', '"this/last month"': r'this month|last month', 'spaced "N P" id': r'\bN P\b',
        '"followed up ... count"': r'followed up', 'receipt reference': r'receipt'}
print(f"\n{'phrase':26} {'train':>6} {'dev':>5} {'test':>5}")
for name, p in pats.items():
    c = {k: sum(bool(re.search(p, r['messages'][1]['content'], re.I)) for r in v) for k, v in splits.items()}
    print(f"{name:26} {c['train']:6d} {c['dev']:5d} {c['test']:5d}")

exactly the 8 keys, in order                     400/400
priority = SCHEMA rule on its own fields         387/400
txn fields null for account_access/kyc/other     387/400
txn_id is NP + 10 digits (or null)               380/400
txn_date not after Received                      399/400

phrase                      train   dev  test
order number                    0     0    26
convenience fee                 0     0    12
"I am out X"                    0     0    17
"sum involved"                  0     0    15
"this/last month"               0     0    20
spaced "N P" id                 0     0    12
"followed up ... count"         0     0     9
receipt reference               0     0    33


## 9. Collect the repository files

Writes `official/repo/` with `adapter/`, `predictions_test.jsonl`, `cleaning_log.csv` and `logs/` (configs, training
logs and dev predictions/scores of every run, a few MB). Add `README.md`, `report.md`, `data_prep.ipynb` and this
notebook (File → Download → .ipynb, **with outputs**).

In [26]:
REPO = f'{OUT}/repo'
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(f'{OUT}/adapter', f'{REPO}/adapter')
shutil.copy(PRED, f'{REPO}/predictions_test.jsonl')
shutil.copy(CLEAN_LOG, f'{REPO}/cleaning_log.csv')
keep = ('config.json', 'result.json', 'train_log.jsonl', 'dev_pred.jsonl', 'dev_score.json')
def copy_small(src_root, run):
    for p in glob.glob(f'{src_root}/{run}/**/*', recursive=True):
        if os.path.isfile(p) and os.path.basename(p) in keep and '/adapter/' not in p and '/ckpt' not in p:
            dst = f'{REPO}/logs/{os.path.relpath(p, src_root)}'
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(p, dst)
for run in list(WHAT) + ['B1_zeroshot_LENIENT_diagnostic', 'B2_rules_in_prompt_LENIENT_diagnostic',
                         'FINAL_dev_one_at_a_time', 'FINAL_dev_reppen_1.0']:
    copy_small(LOGS, run)
copy_small(RUNS, FINAL)
res_all = pd.read_csv(f'{LOGS}/results.csv')
for p in sorted(glob.glob(f'{LOGS}/*/result.json')) + [f'{RUNS}/{FINAL}/result.json']:
    if os.path.exists(p):
        row = json.load(open(p))
        res_all = pd.concat([res_all[res_all['run'] != row['run']], pd.DataFrame([row])], ignore_index=True)
res_all.to_csv(f'{REPO}/logs/results.csv', index=False)
shutil.copy(f'{OUT}/augmentation_log.csv', f'{REPO}/logs/augmentation_log.csv')
size = sum(os.path.getsize(p) for p in glob.glob(f'{REPO}/**/*', recursive=True) if os.path.isfile(p)) / 2**20
print(f'repo files in {REPO}: {size:.0f} MB')
for p in sorted(glob.glob(f'{REPO}/*')):
    print('  ', os.path.basename(p))
print(f'\nREBUILD_OK={REBUILD_OK} | AUG_OK={AUG_OK} | total notebook time {(time.time()-T_START)/60:.0f} min')

repo files in /content/drive/MyDrive/vima3ya/official/repo: 59 MB
   adapter
   cleaning_log.csv
   logs
   predictions_test.jsonl

REBUILD_OK=True | AUG_OK=True | total notebook time 100 min
